# Bibliotecas e Leitura do Dataframe

In [287]:
# Bibliotecas
from pathlib import Path
import pandas as pd

# Local do arquivo
raiz = Path.cwd().parent
# Lendo o arquivo tratado
df = pd.read_csv(raiz / "data" / "processed" / "cotacoes_limpas.csv")

# convertendo a coluna de data do tipo string para datatime64
df["data"] = pd.to_datetime(df["data"], format="%Y-%m-%d")

# resetando a index e ordenando
df = df.sort_values(["nome_acao", "data"]).reset_index(drop=True)

# Comfirma que a coluna de data é do tipo datatime64
print(f'Formato da coluna {df["data"][0]} : {type(df["data"][0])}')

Formato da coluna 2021-01-04 00:00:00 : <class 'pandas.Timestamp'>


## Coluna de setores

In [288]:
# Nova Coluna para classificar os ativos por setor.
setores = {
    "WEGE3": "Bens Industriais",
    "EMBJ3": "Bens Industriais",
    "ABEV3": "Consumo",
    "RENT3": "Consumo",
    "BBDC4": "Financeiro",
    "ITUB4": "Financeiro",
    "GGBR4": "Material Basico",
    "KLBN11": "Material Basico",
    "VALE3": "Material Basico",
    "EGIE3": "Petroleo Energia",
    "PETR4": "Petroleo Energia",
    "PRIO3": "Petroleo Energia",
}  # 5º setores diferentes ao todo.

df["setor"] = df["nome_acao"].map(setores)

# Conferindo se todas foram atribuídas.
print("Ações sem setor: ", df["setor"].isna().sum())

df.groupby("setor")["nome_acao"].unique()

Ações sem setor:  0


setor
Bens Industriais            [EMBJ3, WEGE3]
Consumo                     [ABEV3, RENT3]
Financeiro                  [BBDC4, ITUB4]
Material Basico     [GGBR4, KLBN11, VALE3]
Petroleo Energia     [EGIE3, PETR4, PRIO3]
Name: nome_acao, dtype: object

**Observação:** Todas foram adicionadas corretamente.
| setor| ativos|
|---|---|
| **Bens_Industriais** | ['EMBJ3', 'WEGE3'] |
| **Consumo** | ['ABEV3', 'RENT3'] |
| **Financeiro** | ['BBDC4', 'ITUB4'] |
| **Material_Basico** | ['GGBR4', 'KLBN11', 'VALE3'] |
| **Petroleo_Energia** | ['EGIE3', 'PETR4', 'PRIO3'] |

### Função de retorno numérico ou percentual

In [289]:
# Formata números para o formato brasileiro:
f_real = lambda x: f"{x:,.2f}".replace(",", "x").replace(".", ",").replace("x", ".")

# Formata percentuais no padrão brasileiro: 0.0213 -> "2,13%"
f_pct = lambda x: "-" if pd.isna(x) else f"{x:.2%}".replace(".", ",")


def aplicar_formatos(df):
    return df.apply(
        lambda col: (
            col.map(f_real)
            if col.name in ["volume_financeiro", "sharpe_empirico"]
            else (
                col.map(f_pct)
                if col.name
                in [
                    "retorno_diario",
                    "retorno_acumulado",
                    "drawdown",
                    "drawdown_maximo",
                    "volatilidade_anual",
                ]
                else col
            )
        )
    )

## Volume Financeiro (em R$)

In [290]:
# Volume financeiro = quantidade de ações negociadas * preço de fechamento
df["volume_financeiro"] = df["volume"] * df["fechamento"]

# Exibindo o volume financeiro.
df[["data", "nome_acao", "volume", "fechamento", "volume_financeiro"]].head().pipe(
    aplicar_formatos
)

,data,nome_acao,volume,fechamento,volume_financeiro
0,2021-01-04,ABEV3,17835200,15.48,"276.088.896,00"
1,2021-01-05,ABEV3,37105800,15.81,"586.642.698,00"
2,2021-01-06,ABEV3,28371600,16.13,"457.633.908,00"
3,2021-01-07,ABEV3,32540100,16.05,"522.268.605,00"
4,2021-01-08,ABEV3,30188800,16.53,"499.020.864,00"


## Retorno Diário

In [291]:
# retorno diário = variação percentual do preço ajustado em relação ao dia anterior
df["retorno_diario"] = df.groupby("nome_acao")["fechamento_ajustado"].pct_change()

# O primeiro dia de cada ação será vazio, pois não há dados anteriores.
print("Retornos vazios:", df["retorno_diario"].isna().sum())
df[["data", "nome_acao", "fechamento_ajustado", "retorno_diario"]].head().pipe(aplicar_formatos)

Retornos vazios: 12


,data,nome_acao,fechamento_ajustado,retorno_diario
0,2021-01-04,ABEV3,11.716108,-
1,2021-01-05,ABEV3,11.965871,"2,13%"
2,2021-01-06,ABEV3,12.208063,"2,02%"
3,2021-01-07,ABEV3,12.147515,"-0,50%"
4,2021-01-08,ABEV3,12.510807,"2,99%"


## Base 100 e retorno acumulado:

In [292]:
# Base 100: divide cada preço pelo PRIMEIRO preço da própria ação e multiplica por 100.
# Assim todas começam em 100 e a comparação fica correta, independente do preço de cada uma.
primeiro_preco = df.groupby("nome_acao")["fechamento_ajustado"].transform("first")
df["base_100"] = df["fechamento_ajustado"] / primeiro_preco * 100

# Retorno acumulado desde 04/01/2021 (ex.: base 150 -> +50%)
df["retorno_acumulado"] = df["base_100"] / 100 - 1

df[["data", "nome_acao", "fechamento_ajustado", "base_100", "retorno_acumulado"]].head().pipe(aplicar_formatos)

,data,nome_acao,fechamento_ajustado,base_100,retorno_acumulado
0,2021-01-04,ABEV3,11.716108,100.000000,"0,00%"
1,2021-01-05,ABEV3,11.965871,102.131788,"2,13%"
2,2021-01-06,ABEV3,12.208063,104.198961,"4,20%"
3,2021-01-07,ABEV3,12.147515,103.682170,"3,68%"
4,2021-01-08,ABEV3,12.510807,106.782958,"6,78%"


## Drawdown

In [293]:
# Pico histórico: o maior preço que a ação atingiu ATÉ cada dia
pico = df.groupby("nome_acao")["fechamento_ajustado"].cummax()

# Drawdown: quanto o preço está abaixo do pico (0% = está no topo; -30% = caiu 30% desde o topo)
df["drawdown"] = df["fechamento_ajustado"] / pico - 1

df[["data", "nome_acao", "fechamento_ajustado", "drawdown"]].head(10).pipe(aplicar_formatos)

,data,nome_acao,fechamento_ajustado,drawdown
0,2021-01-04,ABEV3,11.716108,"0,00%"
1,2021-01-05,ABEV3,11.965871,"0,00%"
2,2021-01-06,ABEV3,12.208063,"0,00%"
3,2021-01-07,ABEV3,12.147515,"-0,50%"
4,2021-01-08,ABEV3,12.510807,"0,00%"
5,2021-01-11,ABEV3,12.185359,"-2,60%"
6,2021-01-12,ABEV3,12.715156,"0,00%"
7,2021-01-13,ABEV3,12.238339,"-3,75%"
8,2021-01-14,ABEV3,12.365107,"-2,75%"
9,2021-01-15,ABEV3,12.129364,"-4,61%"


## Médias Móveis

In [294]:
# SMA = média simples dos últimos N pregões, calculada separadamente para cada ação
df["sma_21"] = df.groupby("nome_acao")["fechamento_ajustado"].transform(lambda s: s.rolling(21).mean())
df["sma_50"] = df.groupby("nome_acao")["fechamento_ajustado"].transform(lambda s: s.rolling(50).mean())

# Os primeiros 20 dias (SMA 21) e 49 dias (SMA 50) de cada ação ficam vazios: ainda não há dias suficientes
print("SMA 21 vazias:", df["sma_21"].isna().sum(), "(esperado: 12 x 20 = 240)")
print("SMA 50 vazias:", df["sma_50"].isna().sum(), "(esperado: 12 x 49 = 588)")

SMA 21 vazias: 240 (esperado: 12 x 20 = 240)
SMA 50 vazias: 588 (esperado: 12 x 49 = 588)


## Conferência geral nos resultados:

In [295]:
# Drawdown nunca pode ser positivo
print("Maior drawdown encontrado (deve ser 0):", df["drawdown"].max())

# Resultado final de cada ação no período
resumo = df.groupby("nome_acao").agg(
    setor=("setor", "first"),
    retorno_acumulado=("retorno_acumulado", "last"),
    drawdown_maximo=("drawdown", "min"),
).sort_values("retorno_acumulado", ascending=False)

resumo.pipe(aplicar_formatos)

Maior drawdown encontrado (deve ser 0): 0.0


,setor,retorno_acumulado,drawdown_maximo
nome_acao,,,
PETR4,Petroleo Energia,"228,69%","-39,10%"
PRIO3,Petroleo Energia,"208,32%","-34,32%"
EMBJ3,Bens Industriais,"167,50%","-58,27%"
GGBR4,Material Basico,"30,79%","-33,93%"
EGIE3,Petroleo Energia,"25,35%","-17,65%"
ITUB4,Financeiro,"24,02%","-35,87%"
VALE3,Material Basico,"18,11%","-41,34%"
WEGE3,Bens Industriais,"3,51%","-49,51%"
ABEV3,Consumo,"2,57%","-31,42%"


## Volatilidade e Sharpe Empirico

In [296]:
# Volatilidade anualizada: desvio padrão dos retornos diários x raiz de 252 (pregões em um ano)
volatilidade = df.groupby("nome_acao")["retorno_diario"].std() * (252 ** 0.5)
resumo["volatilidade_anual"] = volatilidade

# "Sharpe empírico" pedido no enunciado: retorno acumulado por unidade de risco
# (versão simplificada, sem descontar a taxa livre de risco)
resumo["sharpe_empirico"] = resumo["retorno_acumulado"] / resumo["volatilidade_anual"]

resumo.sort_values("sharpe_empirico", ascending=False).pipe(aplicar_formatos)

,setor,retorno_acumulado,drawdown_maximo,volatilidade_anual,sharpe_empirico
nome_acao,,,,,
PETR4,Petroleo Energia,"228,69%","-39,10%","39,42%","5,80"
PRIO3,Petroleo Energia,"208,32%","-34,32%","46,78%","4,45"
EMBJ3,Bens Industriais,"167,50%","-58,27%","46,82%","3,58"
EGIE3,Petroleo Energia,"25,35%","-17,65%","18,76%","1,35"
GGBR4,Material Basico,"30,79%","-33,93%","35,56%","0,87"
ITUB4,Financeiro,"24,02%","-35,87%","28,83%","0,83"
VALE3,Material Basico,"18,11%","-41,34%","33,90%","0,53"
WEGE3,Bens Industriais,"3,51%","-49,51%","32,83%","0,11"
ABEV3,Consumo,"2,57%","-31,42%","25,46%","0,10"


## Salvando o Arquivo com as métricas.

In [297]:
caminho = raiz / "data" / "processed" / "cotacoes_metricas.csv"
df.to_csv(caminho, index=False)
print(f"Base com métricas salva: {len(df)} linhas | {len(df.columns)} colunas")

Base com métricas salva: 8940 linhas | 16 colunas
